## Research question 4: Geographical map
How do delays distribute geographically across Europe, and which airports stand out as persistent “delay hotspots”?

In the previous research questions, extensive data analysis was conducted. To present these insights in an even more visual and comprehensible manner, the 20 busiest airports in Europe were plotted on a map. This visualisation allows for a direct comparison of airport performance. The size of each point represents the total number of flights, while the color indicates the total delay in minutes. When hovering over an airport, detailed information about the delay at that specific airport is displayed. Additionally, a time slider has been incorporated, enabling users to easily observe how airport traffic and corresponding delays change over time.

In [2]:
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [4]:
## collecting all of the neccesary data from previous research questions
df_AID_TOP20_YMA = pd.read_csv("datasets_rq4/df_AID_TOP20_YMA.csv")
df_delay_type = pd.read_csv("datasets_rq4/df_delay_type.csv")
df_top20 = pd.read_csv("datasets_rq4/df_top20.csv")

## Collecting new data needed for the map
# Locations of the airports
df_Ageo = pd.read_csv("datasets/Airport_Geo2.csv", quoting=3, on_bad_lines="skip")
df_Ageo_top20 = df_Ageo[df_Ageo["ident"].isin(df_top20["APT_ICAO"])]
df_Ageo_top20 = df_Ageo_top20.reset_index()
df_Ageo_top20 = df_Ageo_top20[["ident", "name", "latitude_deg", "longitude_deg", "elevation_ft"]]

world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")


Merging all datasets to one

In [5]:
# changing month column to numbers for df_AID_TOP20_YMA
df_AID_TOP20_YMA['Month_Lobt_num'] = pd.to_datetime(df_AID_TOP20_YMA['Month_Lobt'], format='%B').dt.month


# merging the year and month column 
df_AID_TOP20_YMA['month_year'] = (
    df_AID_TOP20_YMA['Year_Lobt'].astype(str) + '-' +
    df_AID_TOP20_YMA['Month_Lobt_num'].astype(str).str.zfill(2))

df_delay_type['month_year'] = df_delay_type['Year'].astype(str) + '-' + df_delay_type['Month'].astype(str).str.zfill(2)


# merging df_Ageo and df_AT
df_total0 = pd.merge(df_AID_TOP20_YMA, df_Ageo_top20,
                    left_on='APT_ICAO', right_on='ident', how='left')

# merging df_delay_type
df_total1 = pd.merge(df_total0, df_delay_type,
                     left_on=["APT_ICAO", "month_year"], right_on=["APT_ICAO", "month_year"], how="left")

# Sorting by date
df_total1 = df_total1.sort_values('month_year').reset_index(drop=True)


## Code for the map

In [6]:
# Min and Max delay, so the colorscheme stays consistent during the months
min_delay = df_total1['Total Delay (TD)'].min()
max_delay = df_total1['Total Delay (TD)'].max()

# Code for the map
map_fig = px.scatter_map(
    df_total1,
    lat="latitude_deg",
    lon="longitude_deg",
    size="Total_Flights_Period",
    color="Total Delay (TD)",
    color_continuous_scale=["green", "yellow", "red"],
    range_color=[min_delay, max_delay],
    map_style="carto-positron",
    zoom=4,
    width=1000,
    height=700,
    animation_frame="month_year",
    size_max=40,
    hover_name="name", 
    hover_data={
        "Total_Flights_Period": True,
        "Total Flights (TF)": True,
        "Total Delay (TD)": True,
        "Avg Delay per Movement (in min)": True,
        "Delay_Ratio": True,
        "delay_type": True},
)

# Chaning the hover text
hover_text=("<b>%{hovertext}</b><br>" +
    "Total Flights: %{customdata[0]}<br>" +
    "Total Flights delayed: %{customdata[1]}<br>" +
    "Total Delay: %{customdata[2]} minutes <br>" +
    "Average delay per Movement: %{customdata[3]:.2f} minutes <br>" +
    "Delay ratio: %{customdata[4]:.2f} % <br>" +
    "Most common cause of delay: %{customdata[5]} <br>")

map_fig.update_traces(hovertemplate=hover_text)

# Ensuring hover_text is applied for all animation frames
for frame in map_fig.frames:
    for trace in frame.data:
        trace.hovertemplate = hover_text

map_fig.show()


This map shows how airport traffic and delays change over time. It illustrates how airport performance varies between winter and the summer holiday season, when air travel activity increases significantly. In summer, more airports appear in red, indicating longer delays. The dots are also larger, reflecting a higher number of flights, particularly in southern Europe, where many popular holiday destinations are located. This correlation is logical: as the number of arriving and departing flights increases, total delays also tend to rise. When hovering over an airport, key information about its performance is displayed.
